# Vertical gradients — where the centred stencil loses information

The vertical counterpart of
[`field_validation_sparkle.ipynb`](field_validation_sparkle.ipynb).
That notebook took apart the HORIZONTAL artifact — interpolating a
finite difference across the axis it was differenced along.  This one
takes apart the vertical one, which is not the same thing and needs
its own argument.

| | |
|---|---|
| Domain | one 720 × 720 × 51 tile (Section 1) |
| Applies to | every field downstream of `∂/∂z`: N², Ri, vertical_shear, Fr, Bu, R_ib, ertel_pv |
| Companions | `docs/Gradients.md`, `prompts/field_validation_depth.md` §4b |

## The claim, and the correction to it

`vertical_helpers._vertical_derivative` is **centred** at interior
levels:

    (f[k+1] − f[k−1]) / (z[k+1] − z[k−1])

On even spacing that is identically the mean of the two one-sided
slopes either side of level k.  So it never reads level k, and the
first thing anyone notices is that this looks exactly like the
horizontal sparkle with the interpolation baked into the stencil.

**That framing is half wrong, and the half that is wrong matters.**
Averaging two one-sided slopes does two different things depending on
their signs:

- **Same sign, different magnitude** — a monotone *kink*, e.g. the base
  of the mixed layer where the gradient goes from ~0 above to large
  below.  The centred form returns roughly their mean.  That is a
  correct second-order estimate at a place where the derivative is
  genuinely not well defined.  **Not an error.**
- **Opposite signs** — level k is a vertical *extremum*: an inversion,
  a spike, a one-level step.  The two slopes partly annihilate, the
  centred form collapses toward zero while both one-sided slopes stay
  large.  **This is the artifact**, and it is the real vertical
  analogue of the sparkle.

A metric that does not separate these two overstates the problem
badly, because the mixed-layer base is a kink almost everywhere and
will light up the whole map.  Sections 3 and 4 measure them apart.


## Section 1 — Setup: one tile, full water column

In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0
# ------------------------------------------------------------------------

import dask
import numpy as np
import xarray as xr

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
from dbof.preprocessing import vertical_helpers as VH
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile

# tile_utils sets the Agg backend on import; restore inline afterwards.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)
tile = rect_ij_to_tile(
    *tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3))
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}")

ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(
    S3, DATE, tile, ["Theta", "Salt"])
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)

# Depth coordinate, positive downward, and the layer thicknesses.
Z = np.asarray(VH._get_depth_coord(ds_merge).values, dtype=float)
print(f"levels : {len(Z)}, {Z[0]:.1f} m to {Z[-1]:.1f} m")

## Section 2 — The mechanism, on one synthetic column

Before touching model data: three profiles where the right answer is
known by construction.  If the diagnostic cannot separate these, it
cannot be trusted on the tile.


In [ ]:
# Three profiles on the real model levels.
rho_lin = 1025.0 + 0.004 * Z                       # smooth, no kink
rho_kink = 1025.0 + 0.02 * np.clip(Z - 120.0, 0, None) / 10.0
rho_spike = rho_lin.copy()
rho_spike[20] += 0.35                              # one-level inversion


def stencil_1d(rho):
    """centred, the two one-sided slopes, asym and sign flip."""
    n = len(rho)
    c = np.full(n, np.nan)
    c[1:-1] = (rho[2:] - rho[:-2]) / (Z[2:] - Z[:-2])
    above = np.full(n, np.nan)
    below = np.full(n, np.nan)
    above[1:] = (rho[1:] - rho[:-1]) / (Z[1:] - Z[:-1])
    below[:-1] = (rho[1:] - rho[:-1]) / (Z[1:] - Z[:-1])
    bigger = np.where(np.abs(above) >= np.abs(below), above, below)
    with np.errstate(invalid="ignore", divide="ignore"):
        asym = 1.0 - np.abs(c) / np.abs(bigger)
    flip = (above * below) < 0
    return c, bigger, asym, flip


print(f"{'profile':<12}{'max asym':>10}{'sign flips':>12}  verdict")
print("-" * 62)
for nm, r in (("linear", rho_lin), ("kink", rho_kink),
              ("spike", rho_spike)):
    c, b, a, fl = stencil_1d(r)
    print(f"{nm:<12}{np.nanmax(a):>10.2f}{int(np.nansum(fl)):>12}  "
          + ("clean" if np.nanmax(a) < 0.05 else
             ("CANCELLATION" if fl.any() else "curvature only")))

The kink should show a large asymmetry and **zero** sign flips — the
stencil is averaging across a curvature, which is what a centred
difference is supposed to do.  The spike should show sign flips.  That
is the whole distinction, and everything below rests on it.


## Section 3 — The same diagnostic on the tile

`dfig.vertical_stencil_ab` computes all four fields lazily on the full
3D volume, then they are reduced to the four depth levels the pipeline
actually stores.


In [ ]:
rho = CF.potential_density(ds_merge)
mld = CFAD.mixed_layer_depth(ds_merge)

AB = dfig.vertical_stencil_ab(rho, ds_merge)
ab = dfig.pack_tile_levels(
    dfig.compute_levels(AB, ds_merge, mld=mld, levels=LEVELS),
    XC, YC, edge_margin=0, land_mask=LAND, levels=LEVELS,
    verbose=False)

dfig.depth_map_grid(
    ["dz_centred", "dz_onesided", "dz_asym", "dz_signflip"], ab,
    CMAP_CFG, region=REGION, levels=LEVELS,
    diverging_cmaps=DIVERGING, zoom_half_km=ZOOM_HALF_KM,
    suptitle=("Figure 1 — centred vs one-sided ∂ρ/∂z, with curvature "
              "(asym) and cancellation (signflip) separated"))
plt.show()

In [ ]:
print(f"{'level':<10}{'median asym':>13}{'asym>0.25':>11}"
      f"{'SIGN FLIP':>11}")
print("-" * 45)
for lev in LEVELS:
    a = ab["dz_asym"][lev][2]
    sf = ab["dz_signflip"][lev][2]
    if not np.isfinite(a).any():
        continue
    print(f"{lev:<10}{np.nanmedian(a):>13.3f}"
          f"{100 * np.nanmean(a > 0.25):>10.1f}%"
          f"{100 * np.nanmean(sf > 0.5):>10.1f}%")
print("")
print("asym  = curvature, benign at a kink.")
print("SIGN FLIP = true cancellation -- the number that matters.")

**Read the last column.**  A high `asym` with a low sign-flip rate at
the `at MLD` level is the signature of a kink, not of a defect: the
mixed-layer base is a curvature feature almost everywhere on the tile,
so the centred stencil is correctly reporting the mean slope across
it.  Cancellation, if it is anywhere, tends to be nearer the surface
where inversions and one-level steps live.

If the sign-flip rate is a few percent at 25 m and under a percent at
the MLD, the honest conclusion is that the vertical stencil is **not**
the dominant error in the depth pipeline, and blotchiness in the `at
MLD` rows has another cause — see
[`mixed_layer_depth.ipynb`](mixed_layer_depth.ipynb).


## Section 4 — Where the sign flips actually are, in depth

The level-reduced maps only sample four depths.  This is the full
column: what fraction of the tile has a sign flip at each of the 51
model levels.


In [ ]:
sf3d = AB["dz_signflip"]
asym3d = AB["dz_asym"]
frac_flip, med_asym = dask.compute(
    sf3d.mean(dim=[d for d in sf3d.dims if d != "k"]),
    asym3d.median(dim=[d for d in asym3d.dims if d != "k"]),
)
frac = np.asarray(frac_flip.values, dtype=float)
med = np.asarray(med_asym.values, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(10, 5.5), sharey=True)
axes[0].plot(100 * frac, Z, color="#D55E00", linewidth=2)
axes[0].set_xlabel("% of tile with a sign flip", fontsize=9)
axes[0].set_ylabel("depth (m)", fontsize=9)
axes[1].plot(med, Z, color="#0072B2", linewidth=2)
axes[1].set_xlabel("median stencil asymmetry", fontsize=9)
for ax in axes:
    ax.set_ylim(Z.max(), 0)
    ax.grid(alpha=0.25, linewidth=0.6)
fig.suptitle("Figure 2 — cancellation and curvature vs depth "
             "(surface at top)", fontsize=12)
fig.tight_layout()
plt.show()

k_worst = int(np.nanargmax(frac))
print(f"worst level for cancellation: k={k_worst} "
      f"(z = {Z[k_worst]:.0f} m), {100 * frac[k_worst]:.1f}% of the tile")
print(f"worst level for curvature   : k={int(np.nanargmax(med))} "
      f"(z = {Z[int(np.nanargmax(med))]:.0f} m)")

## Section 5 — What this means for the subset notebooks

Fill this in from the numbers above rather than from the prior
expectation.  The two questions worth answering:

1. **Is the sign-flip rate large enough to matter?**  If it is a few
   percent, then `Ri`, `Fr`, `Bu` and `ertel_pv` will show isolated
   extreme pixels rather than systematic bias, and the right response
   is to note them, not to change the stencil.
2. **Is there a depth band where it concentrates?**  Figure 2 answers
   this directly.  If cancellation peaks well above the pycnocline, it
   is surface inversions, and it will contaminate the `_sfc` and
   `_z25m` rows rather than `_mld`.

### If a fix is ever warranted

The vertical grid is staggered: `k_l` levels sit between tracer levels.
A one-sided difference lands naturally on `k_l`, so the vertical
analogue of the horizontal square-before-interp fix exists — compute
`∂ρ/∂z` on `k_l` and only interpolate to `k` after whatever squaring
the consumer needs.  That would change `N2` and every field below it,
so it is not a change to make on the strength of a map.  Make it on
the strength of the number in Section 3.

---

### Cross-references

- **The horizontal artifact** — `docs/Gradients.md`,
  `field_validation_sparkle.ipynb`.
- **The MLD staircase**, which is a different mechanism with similar
  symptoms — `mixed_layer_depth.ipynb`.
- **The fields affected** — `depth_fields/vertical_shear.ipynb`,
  `mixing_parameters.ipynb`, `ertel_pv.ipynb`.
